RQ3 failure modes (exploratory, no test, no threshold).

Sample: `draw_failure_modes`, n = 30 misclassified primary verification pins per model (invalid = wrong), equal per bin, fixed seed.
Pass 1: open code per item (free text). Pass 2: group open codes into categories (`CODEBOOK`). Single coder.
Log: `gold/failure_modes.jsonl`.

In [103]:
%env DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib

import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from affine import Affine
from IPython.display import display
from rasterio.windows import Window
from shapely import from_wkb

from thesis.data import naip
from thesis.data.eligibility import context_for_scene
from thesis.eval.scorer import predictions
from thesis.eval.subsamples import draw_failure_modes
from thesis.render.marker import render_at_tip
from thesis.render.presets import load_presets
from thesis.run import REPO_ROOT, _geometry_to_crs, _valid_context_footprints, load_config

out_dir = REPO_ROOT / "data" / "dataset_v2"
imagery_config = load_config("imagery")
scene_cache = REPO_ROOT / imagery_config["scene_cache_dir"]
preset = load_presets(load_config("render"))["google_pin_red"]
scoring = load_config("scoring")
CODER = "phillip"

env: DYLD_FALLBACK_LIBRARY_PATH=/opt/homebrew/lib


In [104]:
excluded = {entry["source_id"] for entry in scoring["excluded_sources"]}
sources = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")[["pin_id", "crop_window", "crop_transform"]]
boxes = gpd.read_parquet(out_dir / "inventory" / "boxes.parquet")
box_geometries = boxes.set_index("box_id").geometry.to_dict()
footprints = _valid_context_footprints(gpd.read_parquet(REPO_ROOT / load_config("overture")["out"]))

answers = pd.read_parquet(out_dir / "main_run" / "answers.parquet")
answers["inside"] = answers["bin_id"] < 0
verification = answers.loc[
    (answers["purpose"] == "primary")
    & (answers["task"] == "verification")
    & answers["bin_id"].isin(scoring["primary_bins"])
    & ~answers["source_id"].isin(excluded)
]
scored = predictions(verification, scoring["invalid_policy"])
errors = scored.loc[scored["predicted_inside"] != scored["inside"]]

sample = (
    draw_failure_modes(errors)
    .merge(pins, on="pin_id")
    .merge(
        sources[["source_id", "box_id", "naip_item_id", "scene_window", "naip_source_crs", "scene_geometry_wkb"]],
        on="source_id",
    )
    .sample(frac=1, random_state=0)  # shuffle so models and bins are interleaved
    .reset_index(drop=True)
)
errors.groupby(["model", "bin_id"]).size().unstack(), sample.groupby(["model", "bin_id"]).size().unstack()

(bin_id      -5    -3     3    5    10   20   40   80
 model                                               
 anthropic  1080  1265   420  283  205  139   77   35
 google      185   359   619  300   98   52   31   11
 openai      198   295  1120  777  287   78   38   21,
 bin_id     -5   -3    3    5    10   20   40   80
 model                                            
 anthropic    4    4    4    4    4    4    3    3
 google       4    4    4    4    4    4    3    3
 openai       4    4    4    4    4    4    3    3)

In [105]:
# same crop loading as rq1_human
item_records: dict[str, dict[str, dict]] = {}


def get_item(box_id: str, naip_item_id: str) -> dict:
    if box_id not in item_records:
        box_wgs84 = _geometry_to_crs(box_geometries[box_id], "EPSG:5070", "EPSG:4326")
        item_records[box_id] = {
            str(item["id"]): item for item in naip.search_items_for_geometry(box_wgs84, imagery_config)
        }
    return item_records[box_id][str(naip_item_id)]


def raw_crop_for(row) -> np.ndarray:
    item = get_item(str(row["box_id"]), row["naip_item_id"])
    scene_window = Window(*[int(v) for v in row["scene_window"]])
    cache_key = naip.window_cache_key(str(row["naip_item_id"]), scene_window)
    tile = naip.load_cached(scene_cache, cache_key)
    if tile is None:
        tile = naip.read_item_window(
            item, scene_window, float(imagery_config["gsd_m"]), float(imagery_config["gsd_tolerance"])
        )
        naip.store_atomic(scene_cache, cache_key, tile.array, tile.metadata)
    col_off, row_off, width, height = [int(v) for v in row["crop_window"]]
    return tile.array[:, row_off : row_off + height, col_off : col_off + width]


def display_item(row) -> None:
    # left: what the model saw, right: + Overture footprints
    rendered = render_at_tip(raw_crop_for(row), preset)
    scene_wgs84 = _geometry_to_crs(from_wkb(row["scene_geometry_wkb"]), "EPSG:5070", "EPSG:4326")
    scene_footprints = context_for_scene(footprints, scene_wgs84).to_crs(str(row["naip_source_crs"]))
    to_px = ~Affine(*row["crop_transform"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax in axes:
        ax.imshow(np.transpose(rendered, (1, 2, 0)))
        ax.axis("off")
    for geometry in scene_footprints.geometry:
        for polygon in getattr(geometry, "geoms", [geometry]):
            cols, rows = zip(*[to_px * xy for xy in polygon.exterior.coords])
            axes[1].plot(np.array(cols) - 0.5, np.array(rows) - 0.5, color="yellow", linewidth=1)
    label = "inside (on roof)" if row["inside"] else "outside"
    fig.suptitle(
        f"{row['model']}  bin {row['bin_id']}  exact {row['exact_signed_distance_m']:.2f} m  label: {label}  model: {row['parsed_answer']}"
    )
    display(fig)
    plt.close(fig)

In [106]:
log_path = out_dir / "gold" / "failure_modes.jsonl"
done = set()
if log_path.exists():
    done = {(r["model"], r["pin_id"]) for r in map(json.loads, log_path.read_text().splitlines())}
remaining = sample.loc[[(m, p) not in done for m, p in zip(sample["model"], sample["pin_id"])]].reset_index(drop=True)
i = 0
len(remaining)

90

In [197]:
# run repeatedly
row = remaining.iloc[i]
display_item(row)
print(f"[{i + 1}/{len(remaining)}]  state: {row['refusal_error_state']}")
print(row["raw_text"])

open_code = input("open code: ").strip()
note = input("note (optional): ").strip()
with log_path.open("a") as f:
    f.write(
        json.dumps(
            {
                "model": row["model"],
                "pin_id": row["pin_id"],
                "source_id": row["source_id"],
                "bin_id": int(row["bin_id"]),
                "coder": CODER,
                "open_code": open_code,
                "note": note,
            }
        )
        + "\n"
    )
i += 1

IndexError: single positional indexer is out-of-bounds

In [202]:
# pass 2: open code -> category, filled after all items are coded
CODEBOOK = {
    "no visible cause": "no visible cause",
    "edge localization": "edge localization",
    "trees": "partial occlusion from nature",
    "other structures": "other structures",
    "regitration": "registration",
    "low contrast": "low contrast rooftop",
    "footprint registration": "registration",
    "edge localization, contrast": "low contrast rooftop",
    "trees, lake": "partial occlusion from nature",
    "mistake": "no visible cause",
    "footprint misaligned": "registration",
    "porch roof not part of footprint": "registration",
    "outbuilding": "other structures",
    "lake": "partial occlusion from nature",
    "marker occlusion": "marker occlusion",
    "registration": "registration",
}
codes = pd.read_json(log_path, lines=True)
codes["open_code"].value_counts()

open_code
no visible cause                    21
edge localization                   16
trees                               16
other structures                    13
marker occlusion                     8
registration                         6
low contrast                         2
footprint registration               1
edge localization, contrast          1
trees, lake                          1
mistake                              1
footprint misaligned                 1
porch roof not part of footprint     1
outbuilding                          1
lake                                 1
Name: count, dtype: int64

In [203]:
codes["category"] = codes["open_code"].map(CODEBOOK)
assert codes["category"].notna().all(), sorted(set(codes.loc[codes["category"].isna(), "open_code"]))
pd.crosstab(codes["category"], codes["model"], margins=True), pd.crosstab(codes["category"], codes["bin_id"])

(model                          anthropic  google  openai  All
 category                                                     
 edge localization                      8       4       4   16
 low contrast rooftop                   0       2       1    3
 marker occlusion                       1       4       3    8
 no visible cause                      12       4       6   22
 other structures                       1       9       4   14
 partial occlusion from nature          5       3      10   18
 registration                           3       4       2    9
 All                                   30      30      30   90,
 bin_id                         -5   -3    3    5    10   20   40   80
 category                                                             
 edge localization                3    2    6    4    1    0    0    0
 low contrast rooftop             2    1    0    0    0    0    0    0
 marker occlusion                 0    0    3    4    1    0    0    0
 no visible ca